In [1]:
import sys
sys.path.append('../..')
sys.path.append('..')

In [2]:
import pathlib
import torch
from modis.utils.data import get_dataloaders, summarize_dataset, SemiSupervisedDataset
from modis.model import MODIS
from modis.utils.config import load_config
from modis.utils.utils import calc_classification_metrics
from src.intersim_dataset import get_datasets

In [213]:
def evaluate_model(
    checkpoint_path: pathlib.Path,
    config_file: pathlib.Path,
    datasets: list[torch.utils.data.Dataset]
) -> dict:
    """Validate the model on labeled samples"""
    config = load_config(config_file)
    device = config.device

    model = MODIS(config)
    model.load_from_checkpoint(checkpoint_file, verbose=False)

    dataloaders = get_dataloaders(datasets, batch_size=config.batch_size, drop_last=False, shuffle=False)  #######

    mse_loss = torch.nn.MSELoss()

    model.eval()
    pred_y = []
    true_y = []
    recon_loss = []
    with torch.no_grad():
        for idx, dl in enumerate(dataloaders):
            for data in dl:
                x, y = data[0].to(device), data[1].to(device)

                label_mask = torch.tensor([True if label != -1 else False for label in y])

                if sum(label_mask) == 0:
                    continue
                
                x = x[label_mask]
                y = y[label_mask]

                if x.size(0) == 0:
                    print(f"[!] No labeled samples found in modality {idx}, skipping")
                    continue

                pred_y.append(model.predict(x, input_modality=idx))
                true_y.append(y.view(-1))

                # Reconstruction
                latents = model.get_latents(x, input_modality=idx)
                reconstruction = model.variational_autoencoders[idx].decode(latents)
                recon_loss.append(mse_loss(reconstruction, x).cpu())

    true_y = torch.cat(true_y, dim=0).tolist()
    pred_y = torch.cat(pred_y, dim=0).tolist()

    print(len(true_y), len(pred_y))
    metrics = calc_classification_metrics(true_labels=true_y, pred_labels=pred_y)

    recon_loss = torch.stack(recon_loss).mean().item()
    metrics['mse'] = recon_loss

    return metrics

In [214]:
train_datasets = get_datasets(
    dataset_name = 'intersim_2_delta',
    pairing = 'unpaired',
    split = 'train',
    data_path = '../data',
    include_sample_ids = False
)
train_datasets = [SemiSupervisedDataset(dataset, labeled_ratio=0.1, random_seed=1234) for dataset in train_datasets]
dataloaders = get_dataloaders(train_datasets, 32, drop_last=False, shuffle=False)
summarize_dataset(dataloaders)

Dataset 0 (3068 samples)
Samples per class: {-1: 2762, 0: 44, 1: 66, 2: 57, 3: 64, 4: 75}

Dataset 1 (3067 samples)
Samples per class: {-1: 2761, 0: 49, 1: 83, 2: 63, 3: 43, 4: 68}

Dataset 2 (3064 samples)
Samples per class: {-1: 2758, 0: 48, 1: 72, 2: 65, 3: 56, 4: 65}

Total samples: 9199: {-1: 8281, 0: 141, 1: 221, 2: 185, 3: 163, 4: 208}
Global labeled samples ratio: 0.1


In [215]:
config_file = '../config/intersim/intersim.yaml'
checkpoint_file = '../saved/checkpoints/intersim_2_delta/intersim/20250717_133741/checkpoint_latest.pth'
# config = load_config(config_file)
# model = MODIS(config)
# model.load_from_checkpoint(checkpoint_file, verbose=False)

In [216]:
evaluate_model(checkpoint_file, config_file, train_datasets)

918 918


{'acc': 0.9368191721132898,
 'inv-freq-w-acc': 0.9352289121307719,
 'bacc': 0.9352289121307719,
 'nmi': 0.8306807256154645,
 'ji': 0.8789191951116729,
 'ari': 0.8551881503802595,
 'f1': 0.9367839885786851,
 'mse': 0.027604753151535988}

In [208]:
9199 - 8281

918